In [1]:
# Python Library
import os
import glob
import sys
import numpy as np

from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy import units as u
from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack
from astropy.table import hstack
import warnings
warnings.filterwarnings("ignore")

# Plot presetting
import matplotlib.pyplot as plt
import matplotlib as mpl

# Jupyter Setting
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')

In [2]:
### Helper Functions
import sys
sys.path.append(os.path.join('..', 'src'))
from helper import makeSpecColors
from paths import *
from var import *
from sdtpy import *

In [3]:
logtxt = ""

In [5]:
input_rubin_table = os.path.join(PHOT_WOLLAEGER_RUBIN_DETECTED_DATA, "synphot_anormaly_class_detection.csv")	
input_7dt_table = os.path.join(PHOT_WOLLAEGER_7DT_DETECTED_DATA, "synphot_anormaly_class_detection.csv")	

rubin_table = Table.read(input_rubin_table)
sdt_table = Table.read(input_7dt_table)

print(f"Rubin Table: {len(rubin_table):,} rows")
print(f"7DT Table: {len(sdt_table):,} rows")

logtxt += f"Rubin Table: {len(rubin_table):,} rows\n"
logtxt += f"7DT Table: {len(sdt_table):,} rows\n"

Rubin Table: 6,593 rows
7DT Table: 2,209 rows


In [6]:
import pandas as pd
from astropy.table import Table

# 1) Astropy → pandas
df_rubin = rubin_table.to_pandas().reset_index().rename(columns={'index':'rubin_idx'})
df_sdt   = sdt_table.to_pandas().reset_index().rename(columns={'index':'sdt_idx'})

# 2) 키 컬럼만 추출해서 inner join → 공통 키만 남긴다
common = pd.merge(
    df_sdt[['spec']], 
    df_rubin[['spec']],
    on=['spec'],
    how='inner'
)

# 3) 이 공통 키를 기준으로 원본 테이블에서 필터링
df_rubin_common = pd.merge(
    df_rubin,
    common,
    on=['spec'],
    how='inner'
)
df_sdt_common = pd.merge(
    df_sdt,
    common,
    on=['spec'],
    how='inner'
)

# 4) 필요하면 다시 Astropy Table로!
rubin_common_table = Table.from_pandas(df_rubin_common.drop(columns=['rubin_idx']))
sdt_common_table   = Table.from_pandas(df_sdt_common  .drop(columns=['sdt_idx']))

logtxt += f"{len(rubin_common_table):,} rows\n"
logtxt += f"{len(sdt_common_table):,} rows\n"

In [7]:

logtxt += f"Number Check: {len(rubin_common_table):,} == {len(sdt_common_table):,} ({len(rubin_common_table) == len(sdt_common_table)})\n"
len(rubin_common_table) == len(sdt_common_table)

True

In [8]:
keys_common = list(set(rubin_common_table.keys()) & set(sdt_common_table.keys()))
key_ignore = [f"{key}_{i}" for key in keys_common for i in (1, 2)] + [
    key for key in sdt_common_table.keys() if any(x in key for x in ("fnu", "magapp", "magerr", "snr", "magabs"))
] + [
    key for key in rubin_common_table.keys() if any(x in key for x in ("fnu", "magapp", "magerr", "snr", "magabs"))
]

print(keys_common)
print(key_ignore)

['flagtype', 'vw', 'md', 'vd', 'ang', 'type', 'detection', 'spec', 'mw', 'phase']
['flagtype_1', 'flagtype_2', 'vw_1', 'vw_2', 'md_1', 'md_2', 'vd_1', 'vd_2', 'ang_1', 'ang_2', 'type_1', 'type_2', 'detection_1', 'detection_2', 'spec_1', 'spec_2', 'mw_1', 'mw_2', 'phase_1', 'phase_2', 'magabs_m400', 'snr_m400', 'magapp_m400', 'magerr_m400', 'fnuobs_m400', 'fnuerr_m400', 'fnu_m400', 'magabs_m412', 'snr_m412', 'magapp_m412', 'magerr_m412', 'fnuobs_m412', 'fnuerr_m412', 'fnu_m412', 'magabs_m425', 'snr_m425', 'magapp_m425', 'magerr_m425', 'fnuobs_m425', 'fnuerr_m425', 'fnu_m425', 'magabs_m437', 'snr_m437', 'magapp_m437', 'magerr_m437', 'fnuobs_m437', 'fnuerr_m437', 'fnu_m437', 'magabs_m450', 'snr_m450', 'magapp_m450', 'magerr_m450', 'fnuobs_m450', 'fnuerr_m450', 'fnu_m450', 'magabs_m462', 'snr_m462', 'magapp_m462', 'magerr_m462', 'fnuobs_m462', 'fnuerr_m462', 'fnu_m462', 'magabs_m475', 'snr_m475', 'magapp_m475', 'magerr_m475', 'fnuobs_m475', 'fnuerr_m475', 'fnu_m475', 'magabs_m487', 'snr_m4

In [10]:
rubin_keys_to_merge = keys_common + [key for key in rubin_common_table.keys() if key not in keys_common and key not in key_ignore]
sdt_keys_to_merge = [key for key in sdt_common_table.keys() if key not in keys_common and key not in key_ignore]

merged_table = hstack([rubin_common_table[rubin_keys_to_merge], sdt_common_table[sdt_keys_to_merge]])
logtxt += f"Number of merged synphot tables: {len(merged_table):,}\n"
merged_table[:3]

flagtype,vw,md,vd,ang,type,detection,spec,mw,phase,magobs_u,magobs_g,magobs_r,magobs_i,magobs_z,magobs_y,magobs_m400,magobs_m412,magobs_m425,magobs_m437,magobs_m450,magobs_m462,magobs_m475,magobs_m487,magobs_m500,magobs_m512,magobs_m525,magobs_m537,magobs_m550,magobs_m562,magobs_m575,magobs_m587,magobs_m600,magobs_m612,magobs_m625,magobs_m637,magobs_m650,magobs_m662,magobs_m675,magobs_m687,magobs_m700,magobs_m712,magobs_m725,magobs_m737,magobs_m750,magobs_m762,magobs_m775,magobs_m787,magobs_m800,magobs_m812,magobs_m825,magobs_m837,magobs_m850,magobs_m862,magobs_m875,magobs_m887
str4,float64,float64,float64,float64,str2,str4,str141,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
blue,0.15,0.001,0.05,15.0,KN,True,/home/gpaek/SED-Classifier/notebook/../data/Spectra/Wollaeger+21/z0.010/kn_spectrum_md0.001_vd0.050_mw0.010_vw0.150_ang15.000_phase0.500.fits,0.01,0.5,21.59074014776342,21.823881425464585,22.230757170910845,22.5456858235801,22.993239840210787,23.212390633367857,21.376,21.792,21.598,21.548,21.744,21.577,21.694,21.917,22.004,21.823,22.12,21.755,21.825,22.034,22.131,22.23,22.583,22.099,22.27,22.645,22.961,22.577,22.467,22.765,22.261,21.917,22.404,23.336,21.912,23.267,22.846,26.201,23.056,18.199,23.811,21.006,25.895,22.832,21.892,23.686
blue,0.15,0.001,0.05,30.0,KN,True,/home/gpaek/SED-Classifier/notebook/../data/Spectra/Wollaeger+21/z0.010/kn_spectrum_md0.001_vd0.050_mw0.010_vw0.150_ang30.000_phase0.500.fits,0.01,0.5,21.53241098814123,21.682547078106204,22.0954837663545,22.33782393268926,22.674452016564945,23.52658288635439,21.537,21.282,21.485,21.578,21.475,21.712,22.074,21.81,21.626,21.759,21.724,21.91,22.133,21.98,21.921,21.845,21.885,21.66,21.826,21.243,22.262,22.334,22.137,23.293,21.984,21.71,21.792,23.428,23.094,21.349,21.554,22.757,25.002,20.713,22.62,23.558,22.023,20.7,21.477,21.289
blue,0.15,0.001,0.05,30.0,KN,True,/home/gpaek/SED-Classifier/notebook/../data/Spectra/Wollaeger+21/z0.010/kn_spectrum_md0.001_vd0.050_mw0.010_vw0.150_ang30.000_phase1.000.fits,0.01,1.0,22.928291346040705,22.13663999205344,22.440503215487272,23.08242942667118,23.03124600147597,24.337759984489416,23.788,22.409,22.261,22.589,21.914,21.816,22.254,21.972,22.031,22.108,21.772,22.276,21.971,22.027,22.437,22.534,22.281,22.555,22.46,21.646,22.359,23.217,22.486,22.703,22.804,23.572,21.905,24.345,25.313,25.002,20.78,18.414,23.583,23.372,20.452,23.894,23.697,23.204,20.62,21.926


In [11]:
merged_table_name = os.path.join(PHOT_DATA, "merged_synphot_anormaly_class.csv")
merged_table.write(merged_table_name, format='csv', overwrite=True)
print(f"Merged table saved to {merged_table_name}")
logtxt += f"Merged table saved to {merged_table_name}\n"
logtxt += "\n"
logtxt += "END\n"
print(logtxt)

Merged table saved to /home/gpaek/SED-Classifier/notebook/../data/Synphot/Photometry/merged_synphot_anormaly_class.csv
Rubin Table: 6,593 rows
7DT Table: 2,209 rows
2,209 rows
2,209 rows
Number Check: 2,209 == 2,209 (True)
Number of merged synphot tables: 2,209
Number of merged synphot tables: 2,209
Merged table saved to /home/gpaek/SED-Classifier/notebook/../data/Synphot/Photometry/merged_synphot_anormaly_class.csv

END



In [12]:
logtxt_path = os.path.join(PHOT_DATA, "log_anomaly.txt")
with open(logtxt_path, 'w') as f:
    f.write(logtxt)